In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("Set2")

print("Libraries loaded successfully!")

In [ ]:

news_df = pd.read_csv('../data/raw/fnspid.csv')  # Adjust path as needed

print(f"Dataset shape: {news_df.shape}")
print(f"Columns: {news_df.columns.tolist()}")
news_df.head()

In [ ]:
# Headline length analysis
news_df['headline_length'] = news_df['headline'].str.len()

print("=== Headline Statistics ===")
print(f"Mean length: {news_df['headline_length'].mean():.1f} chars")
print(f"Median: {news_df['headline_length'].median():.1f}")
print(f"Min: {news_df['headline_length'].min()}")
print(f"Max: {news_df['headline_length'].max()}")

# Distribution plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(news_df['headline_length'], bins=50, edgecolor='black', alpha=0.7)
ax.set_title('Distribution of News Headline Lengths')
ax.set_xlabel('Number of Characters')
ax.set_ylabel('Frequency')
plt.show()

In [ ]:
# Count articles per publisher
publisher_counts = news_df['publisher'].value_counts()

print("=== Top 10 Most Active Publishers ===")
for i, (pub, count) in enumerate(publisher_counts.head(10).items(), 1):
    print(f"{i}. {pub}: {count:,} articles")

# Visualization
fig, ax = plt.subplots(figsize=(12, 6))
top_publishers = publisher_counts.head(10)
colors = plt.cm.viridis(np.linspace(0, 0.8, 10))
bars = ax.barh(range(len(top_publishers)), top_publishers.values, color=colors)
ax.set_yticks(range(len(top_publishers)))
ax.set_yticklabels([p[:30] + '...' if len(p) > 30 else p for p in top_publishers.index])
ax.set_xlabel('Number of Articles')
ax.set_title('Top 10 Most Active Publishers')
ax.invert_yaxis()

# Add value labels
for i, bar in enumerate(bars):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2, 
            f'{int(bar.get_width()):,}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Count articles by stock
stock_counts = news_df['stock'].value_counts()

print("=== Top 10 Most Covered Stocks ===")
for i, (stock, count) in enumerate(stock_counts.head(10).items(), 1):
    print(f"{i}. {stock}: {count:,} articles")

# Visualization
fig, ax = plt.subplots(figsize=(12, 6))
top_stocks = stock_counts.head(15)
colors = plt.cm.plasma(np.linspace(0, 0.8, 15))
bars = ax.bar(range(len(top_stocks)), top_stocks.values, color=colors)
ax.set_xticks(range(len(top_stocks)))
ax.set_xticklabels(top_stocks.index, rotation=45, ha='right')
ax.set_ylabel('Number of Articles')
ax.set_title('Most Covered Stocks in Financial News')
ax.set_yscale('log')
ax.set_ylabel('Number of Articles (log scale)')

# Add value labels
for i, bar in enumerate(bars):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, 
            f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Parse dates
news_df['date'] = pd.to_datetime(news_df['date'], utc=True)

# Extract temporal features
news_df['publish_date'] = news_df['date'].dt.date
news_df['publish_year'] = news_df['date'].dt.year
news_df['publish_month'] = news_df['date'].dt.month
news_df['publish_hour'] = news_df['date'].dt.hour
news_df['publish_dayofweek'] = news_df['date'].dt.dayofweek

# Daily volume
daily_volume = news_df.groupby('publish_date').size()

print("=== Time Series Statistics ===")
print(f"Date range: {news_df['publish_date'].min()} to {news_df['publish_date'].max()}")
print(f"Total days in dataset: {len(daily_volume)}")
print(f"Average articles per day: {daily_volume.mean():.1f}")
print(f"Max articles in a day: {daily_volume.max()} on {daily_volume.idxmax()}")

# Multi-panel time series visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Daily volume over time
axes[0,0].plot(daily_volume.index, daily_volume.values, color='darkgreen', alpha=0.7, linewidth=0.8)
axes[0,0].fill_between(daily_volume.index, daily_volume.values, alpha=0.3, color='green')
axes[0,0].set_title('Daily News Publication Volume')
axes[0,0].set_xlabel('Date')
axes[0,0].set_ylabel('Number of Articles')
axes[0,0].tick_params(axis='x', rotation=45)

# Hourly distribution
hourly_dist = news_df['publish_hour'].value_counts().sort_index()
axes[0,1].bar(hourly_dist.index, hourly_dist.values, color='coral', edgecolor='black')
axes[0,1].set_title('Publication Volume by Hour of Day (UTC-4)')
axes[0,1].set_xlabel('Hour of Day (0-23)')
axes[0,1].set_ylabel('Number of Articles')

# Day of week distribution
day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_dist = news_df['publish_dayofweek'].value_counts().sort_index()
axes[1,0].bar(day_names, dow_dist.values, color='teal', edgecolor='black')
axes[1,0].set_title('Publication Volume by Day of Week')
axes[1,0].set_xlabel('Day')
axes[1,0].set_ylabel('Number of Articles')
axes[1,0].tick_params(axis='x', rotation=45)

# Monthly distribution
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly_dist = news_df['publish_month'].value_counts().sort_index()
axes[1,1].bar(month_names, monthly_dist.values, color='purple', edgecolor='black')
axes[1,1].set_title('Publication Volume by Month')
axes[1,1].set_xlabel('Month')
axes[1,1].set_ylabel('Number of Articles')

plt.tight_layout()
plt.show()

In [ ]:
# Calculate rolling statistics
rolling_mean = daily_volume.rolling(window=7, center=True).mean()
rolling_std = daily_volume.rolling(window=7, center=True).std()

# Identify spikes (3 standard deviations above mean)
spike_threshold = rolling_mean + 3 * rolling_std
spike_days = daily_volume[daily_volume > spike_threshold]

print("=== Volume Spike Detection ===")
print(f"Found {len(spike_days)} days with unusually high publication volume\n")

for date, volume in spike_days.head(10).items():
    normal_avg = rolling_mean[date]
    print(f"📅 {date}: {volume} articles (normal avg: {normal_avg:.0f}, +{((volume - normal_avg)/normal_avg*100):.0f}%)")

# Visualize spikes
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(daily_volume.index, daily_volume.values, alpha=0.6, label='Daily Volume', color='blue')
ax.plot(rolling_mean.index, rolling_mean, color='orange', label='7-day Rolling Mean', linewidth=2)
ax.fill_between(daily_volume.index, spike_threshold, daily_volume.max(), alpha=0.3, color='red', label='Spike Zone')
ax.scatter(spike_days.index, spike_days.values, color='red', s=50, zorder=5, label='Volume Spikes')
ax.set_title('News Volume with Spike Detection')
ax.set_xlabel('Date')
ax.set_ylabel('Number of Articles')
ax.legend()
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
from nltk.corpus import stopwords

# Download stopwords if needed
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

# Add financial-specific stopwords
financial_stops = {'said', 'says', 'will', 'can', 'may', 'would', 'could', 'also', 'one', 'two', 'three'}
stop_words.update(financial_stops)

# TF-IDF Vectorizer
tfidf = TfidfVectorizer(max_features=30, stop_words=list(stop_words), ngram_range=(1,2))
tfidf_matrix = tfidf.fit_transform(news_df['headline'].fillna(''))

# Get top keywords
feature_names = tfidf.get_feature_names_out()
tfidf_scores = tfidf_matrix.sum(axis=0).A1

# Sort and display top keywords
top_indices = tfidf_scores.argsort()[-20:][::-1]

print("=== Top 20 Keywords and Phrases by TF-IDF ===")
print("(Higher score = more important/distinctive)\n")
for i, idx in enumerate(top_indices, 1):
    print(f"{i:2}. {feature_names[idx]:30} (score: {tfidf_scores[idx]:.4f})")

# Visualization
fig, ax = plt.subplots(figsize=(12, 8))
top_keywords = [feature_names[i] for i in top_indices[:15]]
top_scores = [tfidf_scores[i] for i in top_indices[:15]]

y_pos = range(len(top_keywords))
ax.barh(y_pos, top_scores, color='steelblue')
ax.set_yticks(y_pos)
ax.set_yticklabels(top_keywords)
ax.invert_yaxis()
ax.set_xlabel('TF-IDF Score')
ax.set_title('Top 15 Most Distinctive Keywords/Phrases in Headlines')
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 60)
print("EDA SUMMARY REPORT")
print("=" * 60)

print(f"\n📊 DATASET OVERVIEW:")
print(f"   • Total articles: {len(news_df):,}")
print(f"   • Unique stocks: {news_df['stock'].nunique()}")
print(f"   • Unique publishers: {news_df['publisher'].nunique()}")
print(f"   • Date range: {news_df['publish_date'].min()} to {news_df['publish_date'].max()}")

print(f"\n📝 HEADLINE ANALYSIS:")
print(f"   • Average length: {news_df['headline_length'].mean():.1f} characters")
print(f"   • Shortest headline: {news_df['headline_length'].min()} chars")
print(f"   • Longest headline: {news_df['headline_length'].max()} chars")

print(f"\n⏰ TEMPORAL PATTERNS:")
print(f"   • Peak hour: {hourly_dist.idxmax()}:00 ({hourly_dist.max():,} articles)")
print(f"   • Busiest day: {day_names[dow_dist.idxmax()]} ({dow_dist.max():,} articles)")
if len(monthly_dist) > 0:
    print(f"   • Busiest month: {month_names[monthly_dist.idxmax()-1]} ({monthly_dist.max():,} articles)")

print(f"\n📰 PUBLISHER INSIGHTS:")
print(f"   • Most active: {publisher_counts.index[0]} ({publisher_counts.iloc[0]:,} articles)")
print(f"   • Top 5 publishers share: {(publisher_counts.head(5).sum()/len(news_df)*100):.1f}%")

print(f"\n📈 STOCK COVERAGE:")
print(f"   • Most covered: {stock_counts.index[0]} ({stock_counts.iloc[0]:,} articles)")
print(f"   • Top 10 stocks share: {(stock_counts.head(10).sum()/len(news_df)*100):.1f}%")